# INV-M365-G: Instruction Mediation / No Hidden Context

**Invariants proven:** INV-M365-G-001, INV-M365-G-002

**Lemmas referenced:** LEM-M365-G-001-01, LEM-M365-G-002-01

**Phase 1:** docs/math/instruction_contract.md (Sections 3, 7)

## 1. Setup

Eval accepts only (instruction, tenant_state). No other inputs. Persona, identity, action, params taken only from instruction tuple.

In [ ]:
P = {"Admin", "User"}
A_Admin, A_User = {"admin_mutate", "admin_read"}, {"user_read"}

def eval_spec(instruction, tenant_state):
    p, i, a, params = instruction[0], instruction[1], instruction[2], instruction[3]
    if p not in P:
        return None
    if (p == "Admin" and a not in A_Admin) or (p == "User" and a not in A_User):
        return None
    if a in {"admin_read", "user_read"}:
        return ("Success", tenant_state, [])
    if a == "admin_mutate" and p == "Admin":
        return ("Success", tenant_state + 1, [{}])
    return None

x = ("Admin", "id1", "admin_read", {"key": "v"})
tau = 0

## 2. Lemma Execution

G-001: Persona, identity, action, parameters are exactly those in the instruction tuple; no inference or override.

G-002: Outcome (r, tau', audit) is determined solely by (instruction, tenant_state); no hidden context.

In [ ]:
out1 = eval_spec(x, tau)
out2 = eval_spec(x, tau)
assert out1 == out2, "G-002: same (x,tau) -> same output (no hidden input)"

p, i, a, params = x[0], x[1], x[2], x[3]
assert p == "Admin" and i == "id1" and a == "admin_read" and params == {"key": "v"}
print("Lemma: values taken only from instruction tuple.")

## 3. Invariant Verification

**INV-M365-G-001:** Evaluation uses instruction.persona, .identity, .action, .parameters as sole source.

**INV-M365-G-002:** result, new_tenant_state, audit_events computable from (instruction, tenant_state) only.

In [ ]:
def verify_G001(instruction):
    assert len(instruction) == 4
    assert instruction[0] in P
    assert instruction[2] in A_Admin or instruction[2] in A_User

def verify_G002(eval_fn, instruction, tenant_state):
    o1 = eval_fn(instruction, tenant_state)
    o2 = eval_fn(instruction, tenant_state)
    assert o1 == o2, "Output depends only on (instruction, tenant_state)"

verify_G001(x)
verify_G002(eval_spec, x, tau)
verify_G002(eval_spec, ("User", "id2", "user_read", {}), 5)
print("INV-M365-G-001, G-002: pass_conditions hold.")

## 4. Failure Demonstration

If we passed extra context (e.g. time), a non-deterministic or context-dependent implementation would fail G-002. Our eval_spec has no such input.

In [ ]:
for _ in range(5):
    assert eval_spec(x, tau) == eval_spec(x, tau)
print("No hidden context: identical inputs -> identical outputs.")

## 5. Conclusion

INV-M365-G-001 and INV-M365-G-002 hold: persona/identity/action/params from tuple only; outcome from (instruction, tenant_state) only.

In [ ]:
print("VERIFY: INV-M365-G-001, INV-M365-G-002 — all pass.")